# V20 pilot (validation only): MuRIL fine-tune (pinned recipe) + official K-prune masks + repair x calibration language. Tag v20-pilot-freeze

In [ ]:
%%bash
set -e
cd /kaggle/working && rm -rf quantum
git clone -q https://github.com/gourav-ai-engineer/quantum.git
cd quantum && git checkout -q v20-pilot-freeze && git log --oneline -1 | tee /kaggle/working/commit.txt
O=/kaggle/working/v20; mkdir -p $O
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader | tee $O/gpu.txt
pip install -q -r requirements-ci.txt && pip install -q -e . --no-deps
pip uninstall -y -q torchvision torchaudio || true
python scripts/preflight.py check | tee $O/preflight.txt
python -c "import transformers,torch;assert transformers.__version__=='4.51.3';print(torch.__version__,transformers.__version__)" | tee $O/env.txt
# 1. the one pinned fine-tune + teacher gate (weights kept as kernel output for later stages)
[ -f $O/ft/ft_meta.json ] || python scripts/v20_finetune.py --out $O/ft > $O/ft.log 2>&1 || { tail -30 $O/ft.log; exit 1; }
grep -v -i warn $O/ft.log | tail -16
grep -q '"gate_pass": true' $O/ft/ft_meta.json || { echo TEACHER_GATE_FAIL; exit 1; }
# 2. official K-prune (+ Kwon) masks, English calibration, c 0.5, seeds 7 42 77
export MODEL_NAME=google/muril-base-cased BL_SKIP_CKPT=1 BL_TASKS=mnli BL_CONS=0.5 PREP_ARGS=--no-ckpt CKPT_DIR_ROOT=$O/ft
export BL_KWON_SEEDS="7 42 77" BL_KPRUNE_SEEDS="7 42 77"
bash scripts/v15_kaggle.sh bl /tmp/bl $O > $O/bl_stage.log 2>&1 || echo BL_FAILED
tail -5 $O/bl_stage.log; ls -la $O/bl | head
# 3. pilot: repair x calibration language, validation only
PYTHONPATH=scripts python scripts/v20_pilot.py --ckpt $O/ft/mnli --kprune-masks $O/bl/kprune_masks.npz \
  --index-dir /tmp/bl/inputs/indices --output-dir $O/pilot > $O/pilot.log 2>&1 || { tail -30 $O/pilot.log; echo PILOT_FAILED; }
grep -v -i "warn\|xet" $O/pilot.log | cut -c1-200 | tail -40
cd /kaggle/working && tar czf v20_pilot_results.tgz --exclude='v20/ft/mnli' v20 commit.txt 2>/dev/null || true
ls -la /kaggle/working
